# AndinaLog 03B | WMS Orders | Tratamiento v2

El tratamiento conserva los campos Bronze y produce valores tratados, acciones, aptitudes analíticas y destino final.


In [ ]:
from pathlib import Path
import sys,hashlib
import pandas as pd
import numpy as np
ENTORNO='auto';RUTA_PROYECTO_DRIVE='/content/drive/MyDrive/GIAD'
COLUMNAS_BRONZE=["order_id","cliente_id","producto_id","fecha_despacho","centro_distribucion","camion_id","chofer_id","cantidad_solicitada","cantidad_entregada","tiempo_entrega_prometido_hrs","tiempo_entrega_real_hrs","otif_on_time","otif_in_full","otif"]
CENTROS={"Cochabamba","La Paz","Santa Cruz","Oruro","Tarija"}
def encontrar_raiz():
    if ENTORNO=='drive' or (ENTORNO=='auto' and 'google.colab' in sys.modules):
        from google.colab import drive;drive.mount('/content/drive');return Path(RUTA_PROYECTO_DRIVE)
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/'proyecto-integrador/01_diagnostico/andinalog_wms_orders/salidas/andinalog_wms_orders_didactico_v2_diagnosticado.csv').is_file():return p
    raise FileNotFoundError('No se encontró la raíz')
RAIZ=encontrar_raiz();ENTRADA=RAIZ/'proyecto-integrador/01_diagnostico/andinalog_wms_orders/salidas/andinalog_wms_orders_didactico_v2_diagnosticado.csv';RUTA_BRONZE=RAIZ/'datasets/AndinaLog_03B_Bronce/andinalog_wms_orders.csv';SALIDAS=RAIZ/'proyecto-integrador/02_tratamiento/andinalog_wms_orders/salidas'
df=pd.read_csv(ENTRADA,dtype='string',encoding='utf-8-sig',keep_default_na=False);bronze=pd.read_csv(RUTA_BRONZE,dtype='string',encoding='utf-8-sig',keep_default_na=False)
required=['fila_bronze','en_cuarentena',*COLUMNAS_BRONZE,*[x for c in COLUMNAS_BRONZE for x in (f'{c}_en_cuarentena',f'{c}_motivo')]]
assert not set(required)-set(df);pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE],bronze);assert len(df)==len(bronze)
df=df.rename(columns={'en_cuarentena':'en_cuarentena_diagnostico'});original=df.copy(deep=True)
print('Entrada:',len(df),'| Cuarentena diagnóstica:',int(df.en_cuarentena_diagnostico.eq('True').sum()))


## Preparación determinista

Las cantidades y resultados reales posteriores al despacho se conservan para evaluación, pero se excluyen de las predictoras disponibles al momento de despachar.


In [ ]:
df['acciones_tratamiento']='';df['motivos_tratamiento']=''
def anotar(m,a,mot):
    m=pd.Series(m,index=df.index).fillna(False).astype(bool)
    for c,x in [('acciones_tratamiento',a),('motivos_tratamiento',mot)]:
        previo=df.loc[m,c];df.loc[m,c]=previo.where(previo.eq(''),previo+' | ')+x
for c in ['order_id','cliente_id','producto_id','camion_id','chofer_id']:
    df[f'{c}_tratado']=df[c].str.strip().str.upper();m=df[f'{c}_tratado'].ne(df[c]);anotar(m,f'NORMALIZAR_{c.upper()}','Quitar espacios y normalizar mayúsculas')
df['centro_distribucion_tratado']=df.centro_distribucion.str.strip()
iso=pd.to_datetime(df.fecha_despacho,format='%Y-%m-%d %H:%M:%S',errors='coerce');local=pd.to_datetime(df.fecha_despacho,format='%d/%m/%Y %H:%M',errors='coerce');fecha=iso.fillna(local)
df['fecha_despacho_bolivia_tratada']=fecha.dt.strftime('%Y-%m-%d %H:%M:%S').fillna('');df['fecha_normalizada']=iso.isna()&local.notna();anotar(df.fecha_normalizada,'NORMALIZAR_FECHA_DESPACHO','Formato local interpretado como hora de Bolivia')
df['cantidad_solicitada_corregida_texto']=df.cantidad_solicitada.str.strip().str.lower().eq('cincuenta')
solicitud_preparada=df.cantidad_solicitada.str.strip().mask(df.cantidad_solicitada_corregida_texto,'50')
anotar(df.cantidad_solicitada_corregida_texto,'CONVERTIR_CANTIDAD_SOLICITADA_TEXTO','Conversión determinista de cincuenta a 50 unidades')
for c in ['cantidad_solicitada','cantidad_entregada','tiempo_entrega_prometido_hrs','tiempo_entrega_real_hrs','otif_on_time','otif_in_full','otif']:
    origen=solicitud_preparada if c=='cantidad_solicitada' else df[c].str.strip()
    df[f'{c}_tratado']=pd.to_numeric(origen,errors='coerce')
df['cantidad_entregada_ausente']=df.cantidad_entregada.str.strip().eq('')
anotar(df.cantidad_entregada_ausente,'CONSERVAR_FALTANTE_ENTREGA','No imputar cantidad posterior al despacho mediante una etiqueta OTIF')


## Decisión final y aptitud analítica

Una orden puede permanecer en Silver aunque su cantidad entregada esté ausente, pero no podrá participar en indicadores OTIF que requieran esa cantidad.


In [ ]:
firma=pd.util.hash_pandas_object(df[COLUMNAS_BRONZE],index=False);k=df.order_id_tratado;variantes=firma.groupby(k,dropna=False).transform('nunique');conflicto=k.ne('')&k.duplicated(False)&variantes.gt(1);copia=df.duplicated(COLUMNAS_BRONZE,keep='first')&~conflicto
df['motivo_cuarentena_final']=''
def q(m,mot):
    m=pd.Series(m,index=df.index).fillna(False).astype(bool);prev=df.loc[m,'motivo_cuarentena_final'];df.loc[m,'motivo_cuarentena_final']=prev.where(prev.eq(''),prev+' | ')+mot
q(copia,'Copia exacta posterior');q(conflicto,'order_id con atributos contradictorios')
patterns={'order_id':r'ORD-2026-\d{5}','cliente_id':r'CLI-\d{3}','producto_id':r'PROD-\d{3}','camion_id':r'CAM-\d{2}','chofer_id':r'CHO-\d{3}'}
for c,p in patterns.items():q(~df[f'{c}_tratado'].str.fullmatch(p).fillna(False),f'{c} inválido')
q(~df.centro_distribucion_tratado.isin(CENTROS),'Centro inválido');q(fecha.isna(),'Fecha de despacho imposible')
sol=df.cantidad_solicitada_tratado;ent=df.cantidad_entregada_tratado;prom=df.tiempo_entrega_prometido_hrs_tratado;real=df.tiempo_entrega_real_hrs_tratado
q(sol.isna()|sol.le(0)|sol.mod(1).ne(0),'Cantidad solicitada inválida');q(ent.notna()&(ent.lt(0)|ent.mod(1).ne(0)|ent.gt(sol)),'Cantidad entregada inválida');q(prom.isna()|prom.le(0),'Tiempo prometido inválido');q(real.isna()|real.lt(0),'Tiempo real inválido')
for c in ['otif_on_time','otif_in_full','otif']:q(~df[f'{c}_tratado'].isin([0,1]),f'{c} inválido')
q(df.otif_tratado.ne((df.otif_on_time_tratado.eq(1)&df.otif_in_full_tratado.eq(1)).astype(int)),'OTIF incoherente con componentes')
df['apta_kpi_otif']=df.motivo_cuarentena_final.eq('')&ent.notna()&sol.notna()
df['apta_modelo_predespacho']=df.motivo_cuarentena_final.eq('')&fecha.notna()&df.producto_id_tratado.str.fullmatch(r'PROD-\d{3}').fillna(False)&df.camion_id_tratado.str.fullmatch(r'CAM-\d{2}').fillna(False)
df['en_cuarentena_final']=df.motivo_cuarentena_final.ne('');df['decision_tratamiento']=np.where(df.en_cuarentena_final,'CUARENTENA','SILVER')
silver=df.loc[~df.en_cuarentena_final].copy();cuarentena=df.loc[df.en_cuarentena_final].copy()


## Comprobaciones y exportación

Silver y cuarentena final forman una partición exacta del diagnosticado. Los valores Bronze permanecen visibles.


In [ ]:
assert len(df)==len(silver)+len(cuarentena);assert silver.order_id_tratado.is_unique;assert silver.motivo_cuarentena_final.eq('').all();assert cuarentena.motivo_cuarentena_final.ne('').all();pd.testing.assert_frame_equal(df[original.columns],original)
metricas={'filas_entrada':len(df),'filas_silver':len(silver),'filas_cuarentena_final':len(cuarentena),'filas_recuperadas':int((df.en_cuarentena_diagnostico.eq('True')&~df.en_cuarentena_final).sum()),'filas_apta_kpi_otif':int(df.apta_kpi_otif.sum()),'filas_apta_modelo_predespacho':int(df.apta_modelo_predespacho.sum()),'cantidades_solicitadas_cincuenta_convertidas':int(df.cantidad_solicitada_corregida_texto.sum()),'cantidades_entregadas_ausentes':int(df.cantidad_entregada_ausente.sum())}
reporte=pd.DataFrame([{'metrica':k,'valor':v} for k,v in metricas.items()]);SALIDAS.mkdir(parents=True,exist_ok=True);base='andinalog_wms_orders_didactico_v2_';silver.to_csv(SALIDAS/(base+'silver.csv'),index=False,encoding='utf-8-sig');cuarentena.to_csv(SALIDAS/(base+'cuarentena_final.csv'),index=False,encoding='utf-8-sig');reporte.to_csv(SALIDAS/(base+'reporte_calidad.csv'),index=False,encoding='utf-8-sig');print(metricas);display(df.tail())
